In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.layers import Input, Dense, Embedding, SimpleRNN
from tensorflow.keras.models import Model

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/project/data.csv')

print(df.head())
print(df.shape)

df = df.dropna(subset=['review', 'sentiment']).copy()

df['sentiment'] = df['sentiment'].str.lower().str.strip()

df['sentiment'] = df['sentiment'].map({
    'positive': 2,
    'neutral': 1,
    'negative': 0
})

df = df.dropna(subset=['sentiment']).copy()

df['sentiment'] = df['sentiment'].astype(int)

x = df['review']
y = df['sentiment']

print("Sentiment Categories:", sorted(y.unique()))
print("Number of Classes:", y.nunique())

                                              review             model  \
0  The movie was a complete snoozefest.  I regret...  Gemini-1.5-Flash   
1  Seriously, the worst venue I've ever been to. ...  Gemini-1.5-Flash   
2  This event was so poorly organized, it was a j...  Gemini-1.5-Flash   
3  What a disaster of a movie!  The acting was aw...  Gemini-1.5-Flash   
4  I wouldn't recommend this venue to my worst en...  Gemini-1.5-Flash   

  sentiment  
0  negative  
1  negative  
2  negative  
3  negative  
4  negative  
(10147, 3)
Sentiment Categories: [np.int64(0), np.int64(1), np.int64(2)]
Number of Classes: 3


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size)

tokenizer.fit_on_texts(x_train)

x_train_seq = tokenizer.texts_to_sequences(x_train)
x_test_seq = tokenizer.texts_to_sequences(x_test)

In [ ]:
max_length = 200

x_train_pad = pad_sequences(
    x_train_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

x_test_pad = pad_sequences(
    x_test_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print("Training shape:", x_train_pad.shape)
print("Testing shape:", x_test_pad.shape)

Training shape: (8116, 200)
Testing shape: (2029, 200)


In [ ]:
inputs = Input(shape=(max_length,), name="input_Layer")

embedding = Embedding(
    input_dim=vocab_size,
    output_dim=64,
    mask_zero=True,
    name="Embedding_Layer"
)(inputs)

rnn = SimpleRNN(
    units=64,
    activation='tanh',
    name="SimpleRNN_Layer"
)(embedding)

outputs = Dense(
    units=3,
    activation='softmax',
    name="Dense_Layer"
)(rnn)

model = Model(
    inputs=inputs,
    outputs=outputs,
    name="SimpleRNN_Model"
)

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "SimpleRNN_Model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_Layer         │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Embedding_Layer     │ (None, 200, 64)   │    640,000 │ input_Layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 200)       │          0 │ input_Layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ SimpleRNN_Layer     │ (None, 64)        │      8,256 │ Embedding_Layer[… │
│ (SimpleRNN)         │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dense_Layer (Dense) │ (None, 3)         │        195 │ SimpleRNN_Layer[… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 648,451 (2.47 MB)

 Trainable params: 648,451 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    x_train_pad,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

Epoch 1/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 11s 170ms/step - accuracy: 0.6568 - loss: 0.7823 - val_accuracy: 0.8097 - val_loss: 0.5135
Epoch 2/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 9s 145ms/step - accuracy: 0.9261 - loss: 0.2374 - val_accuracy: 0.8818 - val_loss: 0.3221
Epoch 3/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 12s 180ms/step - accuracy: 0.9835 - loss: 0.0771 - val_accuracy: 0.8990 - val_loss: 0.3201
Epoch 4/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 7s 143ms/step - accuracy: 0.9957 - loss: 0.0345 - val_accuracy: 0.9052 - val_loss: 0.3087
Epoch 5/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 9s 175ms/step - accuracy: 0.9982 - loss: 0.0177 - val_accuracy: 0.8941 - val_loss: 0.3729


In [ ]:
loss, accuracy = model.evaluate(x_test_pad, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", round(accuracy, 4))

64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9069 - loss: 0.2957
Test Loss: 0.2957160770893097
Test Accuracy: 0.9069


In [ ]:
predictions = model.predict(x_test_pad)

predicted_classes = np.argmax(predictions, axis=1)

print(classification_report(
    y_test,
    predicted_classes,
    target_names=['negative', 'neutral', 'positive']
))

64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step
              precision    recall  f1-score   support

    negative       0.92      0.91      0.92       662
     neutral       0.91      0.86      0.88       617
    positive       0.89      0.95      0.92       750

    accuracy                           0.91      2029
   macro avg       0.91      0.90      0.91      2029
weighted avg       0.91      0.91      0.91      2029



In [ ]:
reviews = [
    "The movie was amazing and fantastic",
    "This was a terrible experience",
    "The experience was okay, nothing special"
]

reviewseq = tokenizer.texts_to_sequences(reviews)

reviewpad = pad_sequences(
    reviewseq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

predictions = model.predict(reviewpad)

sentiment_map = {
    0: 'negative',
    1: 'neutral',
    2: 'positive'
}

predicted_sentiments = []

for p in predictions:
    predicted_class = np.argmax(p)
    predicted_sentiments.append(sentiment_map[predicted_class])

print("Raw Predictions:", predictions)

print("Predicted Sentiments:")

for i, review in enumerate(reviews):
    print(f"Review: '{review}' -> Sentiment: {predicted_sentiments[i]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Raw Predictions: [[1.30963949e-02 6.43956363e-01 3.42947274e-01]
 [9.49738085e-01 7.95210805e-03 4.23098654e-02]
 [1.71091929e-02 9.82774794e-01 1.15946015e-04]]
Predicted Sentiments:
Review: 'The movie was amazing and fantastic' -> Sentiment: neutral
Review: 'This was a terrible experience' -> Sentiment: negative
Review: 'The experience was okay, nothing special' -> Sentiment: neutral
